In [46]:

import re

import json

# time is used to add a small delay between web requests so that the server is not overloaded
import time

from collections import Counter

from urllib.parse import urljoin

# PyPDF2 is the library used to read PDF files
import PyPDF2

# docx (python-docx) is the library used to read Word (.docx) files
import docx

# requests is used to send HTTP GET requests to websites and APIs
import requests

# BeautifulSoup is used to parse raw HTML text into a searchable tree
from bs4 import BeautifulSoup

import pandas as pd

# Show long text fully inside DataFrame cells instead of cutting it with '...'
pd.set_option("display.max_colwidth", None)



---
## Activity 1: Word Frequency from a Multi-page PDF

**Problem Statement:** From a multi-page PDF, extract the text from all pages, clean the text (remove commas, full stops, quotations, hyphens and dashes), combine it into one string and report the 10 most frequent words.

**Approach:**
1. Open the PDF in binary mode (`"rb"`) and create a `PdfReader` object.
2. Loop over every page, call `extract_text()` and join everything into a single string.
3. Clean the string: remove commas, full stops and quotation marks, and replace hyphens and dashes with a space (so that two words are never glued together).
4. Convert the text to lowercase so that `Data` and `data` are counted as the same word.
5. Split the text into words and count them with `Counter`.
6. Print the 10 most common words.

In [47]:

pdf_path = "sample.pdf"

try:
    # Open the PDF in "rb" (read-binary) mode because a PDF is a binary file, not a plain text file
    with open(pdf_path, "rb") as file:

        # PdfReader parses the PDF and gives us access to all of its pages
        reader = PyPDF2.PdfReader(file)

        # reader.pages behaves like a list of pages, so len() gives the total number of pages
        print("Total pages in the PDF:", len(reader.pages))

        # Empty string that will collect the text of all pages one after another
        extracted_text = ""

        # Loop over every page object in the PDF
        for page in reader.pages:

            # extract_text() rebuilds the readable text of this single page as a string
            page_text = page.extract_text()

            # A scanned (image only) page has no text layer and returns an empty result, so we skip it
            if page_text:

                extracted_text += page_text + " "

    print("\n First 500 characters of the raw text.")
    print(extracted_text[:500])

except FileNotFoundError:
    print("File 'sample.pdf' not found.")

Total pages in the PDF: 3

 First 500 characters of the raw text.
Introduction to Artificial Intelligence and Text
 Processing
 A Sample Document for Natural Language Processing Labs
1. Overview
Artificial Intelligence (AI) is the branch of computer science that builds machines capable of performing
tasks that normally require human intelligence. These tasks include learning, reasoning, problem
solving, perception, and understanding language. Today, AI is used in healthcare, finance, education,
transportation, and entertainment.
Natural Language Processing (NL


In [48]:
#remove characters that are not part of words, such as punctuation marks and quotes. These characters are removed because they do not contribute to the meaning of the text and can interfere with word counting and analysis.
remove_chars = [",", ".", '"', "'", """, """, "'", "'"]

# A space is used instead of deleting them, otherwise "pre-processing" style words or "word - word" would merge wrongly.
space_chars = ["-", "–", "—", "−"]

# Start the cleaning from the raw text so that the original variable stays unchanged
cleaned_text = extracted_text

for ch in remove_chars:
    # Replace that character with an empty string, which deletes it
    cleaned_text = cleaned_text.replace(ch, "")

# Go through every hyphen or dash character
for ch in space_chars:
    cleaned_text = cleaned_text.replace(ch, " ")

cleaned_text = cleaned_text.lower()

words = cleaned_text.split()

print("Total words after cleaning:", len(words))
print("Unique words:", len(set(words)))

Total words after cleaning: 579
Unique words: 354


In [49]:
word_counts = Counter(words)

top_10 = word_counts.most_common(10)

print("Top 10 most frequent words")


# enumerate() gives a counter (starting from 1) together with each (word, count) pair
for rank, (word, count) in enumerate(top_10, start=1):
    print(rank, ".", word, "-", count)

Top 10 most frequent words
1 . the - 24
2 . and - 22
3 . a - 10
4 . is - 9
5 . text - 8
6 . data - 8
7 . to - 7
8 . words - 7
9 . of - 6
10 . can - 6


In [50]:
top_10_df = pd.DataFrame(top_10, columns=["Word", "Frequency"])

# Start the index from 1 instead of 0 so it works as a rank column
top_10_df.index = range(1, len(top_10_df) + 1)

top_10_df

,Word,Frequency
1,the,24
2,and,22
3,a,10
4,is,9
5,text,8
6,data,8
7,to,7
8,words,7
9,of,6
10,can,6


**Observation:** The most frequent words are mostly common function words such as *the*, *and*, *of* and *to*. These are called **stop words**; they appear everywhere but carry very little meaning. In a real NLP pipeline they are removed before analysis, which is covered in the next labs. Punctuation other than the marks asked in the task (for example `:` `;` `!` `?` and brackets) is still attached to some words, and regular expressions in Lab 3 will handle that.

---
## Activity 2: Paragraphs of a Word Document in a DataFrame

**Problem Statement:** Open a multi-paragraph `.docx` file, extract all paragraphs into a list, build a DataFrame with one row per paragraph, add `word_count` and `char_count` columns, and sort by `word_count` to find the 3 longest paragraphs.



In [51]:
docx_path = "sample.docx"

try:
    document = docx.Document(docx_path)

    # Empty list that will store the text of every non-empty paragraph
    paragraph_list = []

    for para in document.paragraphs:

        text = para.text.strip()

        #  "not text" means this paragraph is blank
        if not text:
            continue

        paragraph_list.append(text)

    
    print("Total non-empty paragraphs extracted:", len(paragraph_list))

except FileNotFoundError:
    print("File 'sample.docx' not found.")

Total non-empty paragraphs extracted: 11


In [52]:
# Build a DataFrame from the list: one row per paragraph and a single column named "paragraph"
df_docx = pd.DataFrame({"paragraph": paragraph_list})

# split() breaks a paragraph into words, len() counts them; apply() repeats this for every row
df_docx["word_count"] = df_docx["paragraph"].apply(lambda text: len(text.split()))

df_docx["char_count"] = df_docx["paragraph"].str.len()

df_docx

,paragraph,word_count,char_count
0,Introduction to Natural Language Processing,5,43
1,Natural Language Processing (NLP) is a field of Artificial Intelligence that focuses on the interaction between computers and human language.,20,141
2,It combines computational linguistics with statistical and machine learning models.,10,83
3,Applications of NLP,3,19
4,"NLP is used in many real-world applications, including chatbots, machine translation, and sentiment analysis.",14,109
5,Search engines also rely heavily on NLP techniques to understand user queries.,12,78
6,Text Preprocessing,2,18
7,"Before any analysis, raw text usually needs to be cleaned and normalized.",12,73
8,"Common preprocessing steps include lowercasing, removing punctuation, and tokenization.",9,87
9,Conclusion,1,10


In [53]:
df_sorted = df_docx.sort_values(by="word_count", ascending=False)

top_3_longest = df_sorted.head(3)

print("The 3 longest paragraphs (by word count):")
top_3_longest

The 3 longest paragraphs (by word count):


,paragraph,word_count,char_count
1,Natural Language Processing (NLP) is a field of Artificial Intelligence that focuses on the interaction between computers and human language.,20,141
10,"NLP continues to grow rapidly, powering many of the tools we use every day.",14,75
4,"NLP is used in many real-world applications, including chatbots, machine translation, and sentiment analysis.",14,109


**Observation:** Sorting by `word_count` puts the long explanatory paragraphs at the top, while headings and one-line paragraphs go to the bottom. `char_count` is always larger than `word_count` because it also counts letters, spaces and punctuation. Blank paragraphs were skipped, so they do not create empty rows in the DataFrame.

---
## Activity 3: JSON API to DataFrame and Group-by

**Problem Statement:** Fetch `https://dummyjson.com/users`, parse the `users` array into a DataFrame using the fields `firstName`, `age`, `address.state` and `company.department`. Group by `company.department` and report the average age per department, sorted in descending order.



In [54]:
url = "https://dummyjson.com/users"

# Send an HTTP GET request; timeout=10 stops the program from waiting forever if the server does not reply
response = requests.get(url, timeout=10)

# raise_for_status() raises an error for bad responses (404, 500 ...) 
response.raise_for_status()



# .json() converts the JSON text into a Python dictionary
data = response.json()

print("keys:", list(data.keys()))

# The actual user records are stored as a list under the key "users"
users = data["users"]

print("Number of users received:", len(users))

keys: ['users', 'total', 'skip', 'limit']
Number of users received: 30


In [ ]:
# json_normalize() flattens nested dictionaries:
df_users_full = pd.json_normalize(users)

# Keep only the four columns asked in the task
df_users = df_users_full[["firstName", "age", "address.state", "company.department"]]

df_users.head()

,firstName,age,address.state,company.department
0,Emily,29,Mississippi,Engineering
1,Michael,36,Alabama,Support
2,Sophia,43,Alabama,Research and Development
3,James,46,Pennsylvania,Support
4,Emma,31,Colorado,Human Resources


In [ ]:
# Step 1: group all rows that have the same department
grouped = df_users.groupby("company.department")

avg_age = grouped["age"].mean()

# Step 3: round to 2 decimals and sort so the department with the highest average age
avg_age = avg_age.round(2).sort_values(ascending=False)

# Step 4: convert the Series into a DataFrame with clear column names
avg_age_df = avg_age.reset_index()
avg_age_df.columns = ["Department", "Average Age"]

print("Average age per department (highest to lowest):")
avg_age_df

Average age per department (highest to lowest):


,Department,Average Age
0,Research and Development,43.00
1,Sales,40.00
2,Support,39.20
3,Engineering,34.00
4,Services,34.00
5,Legal,32.33
6,Training,32.00
7,Marketing,31.00
8,Accounting,31.00
9,Human Resources,30.00


**Observation:** `json_normalize` is the easiest way to work with nested JSON because it turns `address.state` and `company.department` into normal DataFrame columns. Grouping by department and taking the mean of `age` gives one row per department, and `sort_values(ascending=False)` shows the department with the oldest average staff first.

---
## Activity 4: Scraping the Population Table from Wikipedia

**Problem Statement:** Fetch and parse the Wikipedia page *List of countries by population (United Nations)* with BeautifulSoup, extract the population table into a DataFrame, clean the population column (remove commas and convert to integers) and report the top 10 most populous countries.

**Approach:**
1. Download the HTML with `requests.get()`. Wikipedia blocks requests that have no `User-Agent`, so a header is added.
2. Parse it with `BeautifulSoup` and find the `wikitable` that has a *Population* column.
3. Read the header cells (`<th>`) and every data row (`<tr>`) to build a list of rows.
4. Create the DataFrame, clean the population strings (`"1,438,069,596"` becomes `1438069596`) and convert them to integers.
5. Remove the *World* total row, sort in descending order and show the top 10.

In [67]:
# URL of the Wikipedia page that contains the population table
url = "https://en.wikipedia.org/wiki/List_of_countries_by_population_(United_Nations)"

# Wikipedia rejects requests without a User-Agent, so we introduce ourselves as a normal browser
headers = {"User-Agent": "Mozilla/5.0 (Student NLP Lab - COMSATS University Islamabad)"}

# Download the page; requests follows redirects automatically if the page was renamed
response = requests.get(url, headers=headers, timeout=15)

# Stop with an error message if the download failed
response.raise_for_status()

# 'html.parser' is Python's built-in parser
soup = BeautifulSoup(response.text, "html.parser")

print("Page title:", soup.title.get_text(strip=True))

Page title: List of countries and dependencies by population (United Nations) - Wikipedia


In [ ]:
# Small helper function that removes footnote marks such as [1] or [a] from a piece of text
def clean_cell(text):
    # re.sub replaces every "[ ... ]" part with nothing, then strip() removes extra spaces
    return re.sub(r"\[.*?\]", "", text).strip()

# Variable that will hold the table we are looking for
target_table = None

for table in soup.find_all("table", class_="wikitable"):

    # Take the text of the first row, which is the header row of this table
    header_text = table.find("tr").get_text()

    # We want the table that has a column with "Population" in its header
    if "Population" in header_text:
        target_table = table
        break

print("Table found:", target_table is not None)

Table found: True


In [ ]:
# Get the (first row of the table)
header_row = target_table.find("tr")

header_cells = header_row.find_all("th")

headers_list = []
for th in header_cells:
    text = th.get_text(" ", strip=True)
    text = clean_cell(text)
    headers_list.append(text)

print("Columns:", headers_list)

table_rows = []

all_rows = target_table.find_all("tr")

data_rows = all_rows[1:]

for row in data_rows:

    cells = row.find_all(["th", "td"])

    # Build the list of cleaned values for this row using a normal loop
    values = []
    for cell in cells:
        text = cell.get_text(" ", strip=True)
        text = clean_cell(text)
        values.append(text)

    # Only keep the row if it has the same number of values as we have headers
    if len(values) == len(headers_list):
        table_rows.append(values)

# Turn the list of rows into a DataFrame, using headers_list as column names
df_pop = pd.DataFrame(table_rows, columns=headers_list)

print("Shape of raw table:", df_pop.shape)
df_pop.head()

Columns: ['Country or territory', 'Population ( 1 July 2022 )', 'Population ( 1 July 2023 )', 'Change (%)', 'UN continental region', 'UN statistical subregion']
Shape of raw table: (238, 6)


,Country or territory,Population ( 1 July 2022 ),Population ( 1 July 2023 ),Change (%),UN continental region,UN statistical subregion
0,World,"8,021,407,192","8,091,734,930",+0.88%,–,–
1,India,"1,425,423,212","1,438,069,596",+0.89%,Asia,Southern Asia
2,China,"1,425,179,569","1,422,584,933",−0.18%,Asia,Eastern Asia
3,United States,"341,534,046","343,477,335",+0.57%,Americas,Northern America
4,Indonesia,"278,830,529","281,190,067",+0.85%,Asia,South-eastern Asia


In [78]:
# The first column contains the country name
country_col = df_pop.columns[0]

# The table has more than one "Population" column (2022 and 2023), so we take the LAST one (the latest year)
population_col = [col for col in df_pop.columns if "Population" in col][-1]

# Print the names so we know which columns are being used
print("Country column   :", country_col)
print("Population column:", population_col)

# Keep only these two columns and make a copy so the original DataFrame is not changed
df_clean = df_pop[[country_col, population_col]].copy()

# Give them simple names
df_clean.columns = ["Country", "Population"]

# The first data row is "World" (the total of all countries), so it is removed
df_clean = df_clean[df_clean["Country"] != "World"]

# str.replace removes the commas: "1,438,069,596" becomes "1438069596"
df_clean["Population"] = df_clean["Population"].str.replace(",", "", regex=False)

# to_numeric converts the strings into numbers; errors="coerce" turns any bad value into NaN instead of crashing
df_clean["Population"] = pd.to_numeric(df_clean["Population"], errors="coerce")

# Drop rows where the conversion failed (NaN) and then convert the column to integers
df_clean = df_clean.dropna(subset=["Population"])
df_clean["Population"] = df_clean["Population"].astype(int)

# Check that the column is really an integer now
print(df_clean.dtypes)

Country column   : Country or territory
Population column: Population ( 1 July 2023 )
Country       object
Population     int64
dtype: object


In [79]:
# Sort by population (largest first), keep 10 rows and reset the index so it starts cleanly
top_10_countries = df_clean.sort_values(by="Population", ascending=False).head(10).reset_index(drop=True)

# Make the index start from 1 so that it works as a rank
top_10_countries.index = top_10_countries.index + 1

# Show the population with commas again so it is easy to read (the stored values stay integers)
print("Top 10 most populous countries")
top_10_countries.style.format({"Population": "{:,}"})

Top 10 most populous countries


,Country,Population
1,India,"1,438,069,596"
2,China,"1,422,584,933"
3,United States,"343,477,335"
4,Indonesia,"281,190,067"
5,Pakistan,"247,504,495"
6,Nigeria,"227,882,945"
7,Brazil,"211,140,729"
8,Bangladesh,"171,466,990"
9,Russia,"145,440,500"
10,Mexico,"129,739,759"


**Observation:** The table needed cleaning before it could be used: the population values were strings with commas, so they were converted to integers, and the *World* row had to be removed because it is not a country. Without the integer conversion, sorting would have been alphabetical (as text) instead of numerical, giving wrong results.

---
## Activity 5: Scraping Quotes and Saving Them as JSON

**Problem Statement:** Scrape `https://quotes.toscrape.com/tag/love/` for the quote text and author, store the results as a list of dictionaries, find which author appears most often, compute the average quote length in characters and save the raw scraped list to a `.json` file.



In [ ]:
base_url = "https://quotes.toscrape.com"
next_page = "/tag/love/"

# List that will store one dictionary per quote
quotes_list = []

while next_page:

    # Join the base address and the page path into a complete URL
    page_url = urljoin(base_url, next_page)

    response = requests.get(page_url, timeout=10)
    response.raise_for_status()

    # Parse the HTML of this page
    soup = BeautifulSoup(response.text, "html.parser")

    for quote_block in soup.find_all("div", class_="quote"):

        text = quote_block.find("span", class_="text").get_text(strip=True)

        # The author's name is inside <small class="author">
        author = quote_block.find("small", class_="author").get_text(strip=True)

        quotes_list.append({"text": text, "author": author})

    next_button = soup.find("li", class_="next")

    next_page = next_button.a["href"] if next_button else None

    # Wait one second before the next request to be polite to the server
    time.sleep(1)

# Show how many quotes were collected and the first one as a sample
print("Total quotes scraped:", len(quotes_list))
print("First quote:", quotes_list[0])

Total quotes scraped: 14
First quote: {'text': '“It is better to be hated for what you are than to be loved for what you are not.”', 'author': 'André Gide'}


In [86]:
# Make a list that contains only the author names
authors = [item["author"] for item in quotes_list]

# Counter counts how many quotes each author has
author_counts = Counter(authors)

# most_common(1) gives the top author as a list with one (author, count) pair; [0] takes that pair
top_author, top_count = author_counts.most_common(1)[0]

# Print the result
print(f"Author who appears most often: {top_author} ({top_count} quotes)")

# Show the counts of all authors, highest first, as a small table
pd.DataFrame(author_counts.most_common(), columns=["Author", "Number of Quotes"])

Author who appears most often: Marilyn Monroe (3 quotes)


,Author,Number of Quotes
0,Marilyn Monroe,3
1,Jane Austen,2
2,André Gide,1
3,Bob Marley,1
4,Elie Wiesel,1
5,Friedrich Nietzsche,1
6,Pablo Neruda,1
7,James Baldwin,1
8,C.S. Lewis,1
9,Alfred Tennyson,1


In [87]:
# The site puts each quote inside curly quotation marks; they are only decoration, so they are stripped before measuring
lengths = [len(item["text"].strip("\u201c\u201d")) for item in quotes_list]

# Average = total number of characters divided by the number of quotes
average_length = sum(lengths) / len(lengths)

# Print the average rounded to 2 decimal places, plus the shortest and longest quote for context
print(f"Average quote length: {average_length:.2f} characters")
print("Shortest quote:", min(lengths), "characters")
print("Longest quote :", max(lengths), "characters")

Average quote length: 261.36 characters
Shortest quote: 40 characters
Longest quote : 1082 characters


In [88]:
# Name of the output file
output_file = "quotes_love.json"

# Open the file in write mode; utf-8 encoding is needed because the quotes contain curly quotation marks
with open(output_file, "w", encoding="utf-8") as f:

    # json.dump writes the list of dictionaries into the file;
    # indent=4 makes it readable and ensure_ascii=False keeps the special characters as they are
    json.dump(quotes_list, f, indent=4, ensure_ascii=False)

print(f"Raw data saved to '{output_file}'")

Raw data saved to 'quotes_love.json'


In [89]:
# Read the file back to confirm that it was saved correctly
with open("quotes_love.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)

# The number of records must match the list we scraped
print("Records in the saved file:", len(loaded))
print("Same as scraped list:", loaded == quotes_list)

Records in the saved file: 14
Same as scraped list: True


**Observation:** The `quote` blocks all follow the same tag-and-class pattern, which is why `find_all("div", class_="quote")` works so well. Following the *Next* button makes the scraper collect the quotes of every page instead of only the first one. Saving the data with `json.dump` keeps the list of dictionaries in exactly the same structure, and reading it back with `json.load` proves that nothing was lost.